# MobileNetV3-UNet — Augmented Cross-Validation

MobileNetV3-Large (ImageNet-pretrained) encoder + a custom U-Net decoder,
trained with 5-fold stratified group cross-validation on a mixed
indoor-floor dataset (ADE20K, Hypersim, SUN RGB-D, NYUv2 subsets, filtered
for floor visibility).

Uses a padding-safe resize (aspect-ratio-preserving `LongestMaxSize` +
`PadIfNeeded` instead of a hard resize) and additional geometric/occlusion
augmentation (scale jitter, crop, small rotation/shear, coarse dropout).

**Result:** IoU 0.7956 ± 0.0066 (mean ± std across 5 folds). See
`docs/benchmark_report.md` for the full metrics table. The best-IoU fold's
checkpoint is the one carried forward into fine-tuning in
`02_finetune_hand_annotated.ipynb`.

## Setup

In [ ]:
# Install versions suitable for current Colab runtimes.
!pip -q install "torch>=2.3" "torchvision>=0.18" \
    albumentations==1.4.15 opencv-python-headless==4.10.0.84 \
    onnx==1.16.2 onnxruntime==1.19.2 tqdm pandas scikit-learn

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import gc
import json
import math
import os
import random
import shutil
import time

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import models
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
N_SPLITS = 5

# Use [0] for a quick smoke test. Use list(range(N_SPLITS)) for complete CV.
FOLDS_TO_RUN = list(range(N_SPLITS))

IMAGE_SIZE = 384
BATCH_SIZE = 8
NUM_WORKERS = 2

import zipfile

ZIP_PATH = "/content/drive/MyDrive/peppermint_floor_seg/prepared_v1/drive-download-20260716T054056Z-1-001.zip"
LOCAL_DATA = Path("/content/peppermint_prepared_v1")

if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

print(f"Extracting {ZIP_PATH} -> {LOCAL_DATA} ...")
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(LOCAL_DATA)

print("Done. Contents:")
for root, dirs, files in os.walk(LOCAL_DATA):
    depth = root.replace(str(LOCAL_DATA), "").count(os.sep)
    if depth <= 1:
        print(f"  {root} ({len(files)} files)")

# Verify against the LOCAL extracted copy, not Drive — this is what training actually reads from.
assert LOCAL_DATA.exists(), f"Extracted data not found: {LOCAL_DATA}"
assert (LOCAL_DATA / "images").exists(), f"Missing images/ under {LOCAL_DATA}"
assert (LOCAL_DATA / "masks").exists(), f"Missing masks/ under {LOCAL_DATA}"
assert (LOCAL_DATA / "metadata" / "dataset_manifest.csv").exists(), f"Missing metadata/dataset_manifest.csv under {LOCAL_DATA}"

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Fix Albumentations / Albucore compatibility

!pip uninstall -y albumentations albucore

!pip install -q --no-cache-dir "albucore==0.0.15"
!pip install -q --no-cache-dir --no-deps "albumentations==1.4.15"

# Restart the Colab runtime so the failed imports are cleared.
import os
import signal

os.kill(os.getpid(), signal.SIGKILL)

In [ ]:
# Verify installation after Colab reconnects

import albumentations as A
import albucore

from albumentations.pytorch import ToTensorV2
from albucore.utils import preserve_channel_dim

print("Albumentations:", A.__version__)
print("Albucore:", albucore.__version__)
print("Import successful.")

In [ ]:
# Configuration cell — run before the transform cell

IMAGE_SIZE = 384
BATCH_SIZE = 8
NUM_WORKERS = 2
SEED = 42

print("IMAGE_SIZE:", IMAGE_SIZE)

In [ ]:
# Mount Drive, restore paths, and copy prepared data to /content

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import subprocess

DRIVE_DATA = Path(
    "/content/drive/MyDrive/peppermint_floor_seg/prepared_v1"
)

LOCAL_DATA = Path(
    "/content/peppermint_prepared_v1"
)

DRIVE_MANIFEST = (
    DRIVE_DATA / "metadata" / "dataset_manifest.csv"
)

assert DRIVE_DATA.exists(), (
    f"Prepared dataset not found: {DRIVE_DATA}"
)

assert DRIVE_MANIFEST.exists(), (
    f"Manifest not found: {DRIVE_MANIFEST}"
)

# Copy from Drive only when the local copy is missing.
if not (
    LOCAL_DATA / "metadata" / "dataset_manifest.csv"
).exists():

    LOCAL_DATA.mkdir(
        parents=True,
        exist_ok=True,
    )

    subprocess.run(
        [
            "rsync",
            "-a",
            "--info=progress2",
            f"{DRIVE_DATA}/",
            f"{LOCAL_DATA}/",
        ],
        check=True,
    )

    print("\nDataset copied to:", LOCAL_DATA)

else:
    print("Local dataset already exists:", LOCAL_DATA)


MANIFEST_PATH = (
    LOCAL_DATA / "metadata" / "dataset_manifest.csv"
)

IMAGE_DIR = LOCAL_DATA / "images"
MASK_DIR = LOCAL_DATA / "masks"

assert MANIFEST_PATH.exists()
assert IMAGE_DIR.exists()
assert MASK_DIR.exists()

print("Manifest:", MANIFEST_PATH)
print("Images:", IMAGE_DIR)
print("Masks:", MASK_DIR)

In [ ]:
# Core imports required by the dataset and dataloaders

from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

print("PyTorch:", torch.__version__)
print("Dataset and DataLoader imported successfully.")

In [ ]:
# Import cross-validation splitter

!pip -q install -U scikit-learn

from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 5
SEED = 42

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

print("StratifiedGroupKFold ready.")

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.35),
    A.HueSaturationValue(p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.1),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
    ToTensorV2(),
])

print("Transforms created successfully.")

class FloorDataset(Dataset):
    def __init__(self, root, rows, transform):
        self.root = Path(root)
        self.image_dir = self.root / "images"
        self.mask_dir = self.root / "masks"
        self.rows = rows.reset_index(drop=True).copy()
        self.transform = transform

        required = {"sample_id", "image_filename", "mask_filename"}
        missing = required - set(self.rows.columns)
        assert not missing, f"Manifest columns missing: {missing}"
        assert len(self.rows) > 0

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        image_path = self.image_dir / row["image_filename"]
        mask_path = self.mask_dir / row["mask_filename"]

        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        if image_bgr is None:
            raise RuntimeError(f"Failed to read image: {image_path}")
        if mask is None:
            raise RuntimeError(f"Failed to read mask: {mask_path}")

        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        mask = (mask > 127).astype(np.float32)

        augmented = self.transform(image=image, mask=mask)

        return (
            augmented["image"],
            augmented["mask"].unsqueeze(0),
            row["sample_id"],
        )

def make_loaders(manifest, fold):
    train_rows = manifest[manifest["fold"] != fold].reset_index(drop=True)
    val_rows = manifest[manifest["fold"] == fold].reset_index(drop=True)

    train_ds = FloorDataset(LOCAL_DATA, train_rows, train_transform)
    val_ds = FloorDataset(LOCAL_DATA, val_rows, val_transform)

    generator = torch.Generator()
    generator.manual_seed(SEED + fold)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=True,
        persistent_workers=NUM_WORKERS > 0,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        persistent_workers=NUM_WORKERS > 0,
    )

    return train_loader, val_loader, train_rows, val_rows

manifest = pd.read_csv(
    LOCAL_DATA / "metadata" / "dataset_manifest.csv"
).sort_values("sample_id").reset_index(drop=True)

required_columns = {
    "sample_id", "dataset", "image_filename", "mask_filename",
    "positive_fraction"
}
missing_columns = required_columns - set(manifest.columns)
assert not missing_columns, f"Manifest columns missing: {missing_columns}"
assert manifest["sample_id"].is_unique

# Keep related SUN/NYU frames in the same fold when group_id is available.
if "group_id" not in manifest.columns:
    manifest["group_id"] = manifest["sample_id"]

manifest["group_id"] = manifest["group_id"].fillna(manifest["sample_id"]).astype(str)
manifest["cv_group"] = (
    manifest["dataset"].astype(str)
    + "::"
    + manifest["group_id"].astype(str)
)

# Balance both source dataset and floor coverage across folds.
# Ranking before qcut avoids duplicate-edge failures.
def coverage_bins(series, bins=5):
    effective_bins = min(bins, len(series))
    return pd.qcut(
        series.rank(method="first"),
        q=effective_bins,
        labels=False,
        duplicates="drop",
    ).astype(int)

manifest["coverage_bin"] = (
    manifest.groupby("dataset", group_keys=False)["positive_fraction"]
    .apply(coverage_bins)
    .reset_index(level=0, drop=True)
)

manifest["stratum"] = (
    manifest["dataset"].astype(str)
    + "_coverage_"
    + manifest["coverage_bin"].astype(str)
)

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

manifest["fold"] = -1

for fold, (_, val_indices) in enumerate(
    splitter.split(
        manifest,
        y=manifest["stratum"],
        groups=manifest["cv_group"],
    )
):
    manifest.loc[val_indices, "fold"] = fold

assert (manifest["fold"] >= 0).all()
assert (
    manifest.groupby("cv_group")["fold"].nunique().max() == 1
), "A group leaked across folds."

fold_file = LOCAL_DATA / "metadata" / f"cv_folds_{N_SPLITS}_seed{SEED}.csv"
manifest.to_csv(fold_file, index=False)

print("Total samples:", len(manifest))
display(pd.crosstab(manifest["fold"], manifest["dataset"]))
display(
    manifest.groupby("fold")["positive_fraction"]
    .agg(["count", "mean", "median"])
)

In [ ]:
# PyTorch model imports

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import models
from torch.utils.data import Dataset, DataLoader, Subset

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Neural-network imports ready.")

In [ ]:
# Missing standard-library import

import gc

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Cleanup complete.")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/peppermint_floor_seg"
)

print("Drive root:", DRIVE_ROOT)

assert DRIVE_ROOT.exists(), (
    f"Drive root not found: {DRIVE_ROOT}. "
    "Mount Google Drive first."
)


## Model, loss, and evaluation

In [ ]:
class ConvBNAct(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

class DecoderBlock(nn.Module):
    def __init__(self, input_channels, skip_channels, output_channels):
        super().__init__()
        self.fusion = ConvBNAct(
            input_channels + skip_channels,
            output_channels,
        )

    def forward(self, decoder_feature, skip_feature):
        decoder_feature = F.interpolate(
            decoder_feature,
            size=skip_feature.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        return self.fusion(
            torch.cat(
                [decoder_feature, skip_feature],
                dim=1,
            )
        )

class MobileNetV3UNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        weights = (
            models.MobileNet_V3_Large_Weights.DEFAULT
            if pretrained
            else None
        )

        backbone = models.mobilenet_v3_large(
            weights=weights
        ).features

        # Discover the last feature map at each spatial resolution.
        probe = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        stage_information = []

        backbone.eval()

        with torch.no_grad():
            for index, layer in enumerate(backbone):
                probe = layer(probe)
                stage_information.append(
                    (
                        index,
                        int(probe.shape[1]),
                        int(probe.shape[2]),
                        int(probe.shape[3]),
                    )
                )

        last_stage_per_size = {}

        for item in stage_information:
            last_stage_per_size[(item[2], item[3])] = item

        selected = sorted(
            last_stage_per_size.values(),
            key=lambda item: item[2],
            reverse=True,
        )

        selected = [
            item
            for item in selected
            if item[2] < IMAGE_SIZE
        ][:5]

        assert len(selected) == 5, stage_information

        self.stage_ids = [item[0] for item in selected]
        channels = [item[1] for item in selected]

        self.backbone = backbone
        self.backbone_frozen = False

        c1, c2, c3, c4, c5 = channels

        self.center = ConvBNAct(c5, 256)
        self.decoder4 = DecoderBlock(256, c4, 160)
        self.decoder3 = DecoderBlock(160, c3, 96)
        self.decoder2 = DecoderBlock(96, c2, 64)
        self.decoder1 = DecoderBlock(64, c1, 32)
        self.head = nn.Conv2d(32, 1, kernel_size=1)

        print(
            "Selected MobileNet stages:",
            list(zip(self.stage_ids, channels)),
        )

    def freeze_backbone(self):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

        self.backbone_frozen = True
        self.backbone.eval()

    def unfreeze_backbone(self):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = True

        self.backbone_frozen = False

    def train(self, mode=True):
        super().train(mode)

        # Frozen BatchNorm running statistics must not drift.
        if mode and self.backbone_frozen:
            self.backbone.eval()

        return self

    def decoder_parameters(self):
        modules = [
            self.center,
            self.decoder4,
            self.decoder3,
            self.decoder2,
            self.decoder1,
            self.head,
        ]

        for module in modules:
            yield from module.parameters()

    def forward(self, image):
        input_size = image.shape[-2:]
        features = []
        feature = image
        wanted = set(self.stage_ids)

        for index, layer in enumerate(self.backbone):
            feature = layer(feature)

            if index in wanted:
                features.append(feature)

        assert len(features) == 5

        feature1, feature2, feature3, feature4, feature5 = features

        decoded = self.center(feature5)
        decoded = self.decoder4(decoded, feature4)
        decoded = self.decoder3(decoded, feature3)
        decoded = self.decoder2(decoded, feature2)
        decoded = self.decoder1(decoded, feature1)
        logits = self.head(decoded)

        return F.interpolate(
            logits,
            size=input_size,
            mode="bilinear",
            align_corners=False,
        )

def count_parameters(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    return total, trainable

smoke_model = MobileNetV3UNet(pretrained=True)

with torch.no_grad():
    smoke_output = smoke_model(
        torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
    )

assert smoke_output.shape == (
    2,
    1,
    IMAGE_SIZE,
    IMAGE_SIZE,
)

total_parameters, trainable_parameters = count_parameters(smoke_model)

print("Output shape:", tuple(smoke_output.shape))
print("Total parameters:", total_parameters)
print("Initially trainable:", trainable_parameters)

del smoke_model, smoke_output
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, bce_weight=0.6, eps=1e-6):
        super().__init__()
        self.bce_weight = bce_weight
        self.eps = eps

    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(logits, target)

        probabilities = torch.sigmoid(logits)
        dimensions = (1, 2, 3)

        dice_loss = 1.0 - (
            (
                2.0 * (probabilities * target).sum(dimensions)
                + self.eps
            )
            / (
                probabilities.sum(dimensions)
                + target.sum(dimensions)
                + self.eps
            )
        ).mean()

        return (
            self.bce_weight * bce
            + (1.0 - self.bce_weight) * dice_loss
        )

@torch.no_grad()
def evaluate(model, loader, criterion, threshold=0.5):
    model.eval()

    losses = []
    intersection = 0
    union = 0
    true_positive = 0
    false_positive = 0
    false_negative = 0

    for images, targets, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            enabled=DEVICE.type == "cuda",
        ):
            logits = model(images)
            loss = criterion(logits, targets)

        losses.append(float(loss.item()))

        predictions = torch.sigmoid(logits) >= threshold
        ground_truth = targets >= 0.5

        intersection += int((predictions & ground_truth).sum().item())
        union += int((predictions | ground_truth).sum().item())

        true_positive += int((predictions & ground_truth).sum().item())
        false_positive += int((predictions & ~ground_truth).sum().item())
        false_negative += int((~predictions & ground_truth).sum().item())

    iou = intersection / max(union, 1)
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    dice = (
        2.0 * true_positive
        / max(2.0 * true_positive + false_positive + false_negative, 1)
    )

    return {
        "val_loss": float(np.mean(losses)),
        "iou": float(iou),
        "dice": float(dice),
        "precision": float(precision),
        "recall": float(recall),
    }

def save_json(data, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with open(destination, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2)

## Shared training hyperparameters

Used by the augmented run below (same schedule/LRs as the reference transform pipeline, so the augmentation is the only variable changing).

In [ ]:
# Folds to train

FOLDS_TO_RUN = list(range(N_SPLITS))

print("Folds to run:", FOLDS_TO_RUN)

assert len(FOLDS_TO_RUN) > 0
assert all(
    0 <= fold < N_SPLITS
    for fold in FOLDS_TO_RUN
)

In [ ]:
# Define reproducible seeding before run_mobile_fold()

import os
import random
import numpy as np
import torch


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducible behavior. This can be slightly slower.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

print("Seed function ready. Seed:", SEED)

In [ ]:
# DataLoader configuration — run before make_loaders()

import torch

BATCH_SIZE = 8
NUM_WORKERS = 2

PIN_MEMORY = torch.cuda.is_available()
PERSISTENT_WORKERS = NUM_WORKERS > 0

print("BATCH_SIZE:", BATCH_SIZE)
print("NUM_WORKERS:", NUM_WORKERS)
print("PIN_MEMORY:", PIN_MEMORY)
print("PERSISTENT_WORKERS:", PERSISTENT_WORKERS)

In [ ]:
# Missing progress-bar import

from tqdm.auto import tqdm

print("tqdm imported successfully.")

In [ ]:
import json

# Transfer-learning schedule:
# Phase 1 trains only the randomly initialized U-Net decoder.
# Phase 2 unfreezes the pretrained MobileNet encoder and fine-tunes it.
FROZEN_EPOCHS = 3
FINETUNE_EPOCHS = 12
DECODER_WARMUP_LR = 1e-3
BACKBONE_FINETUNE_LR = 5e-5
DECODER_FINETUNE_LR = 3e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5

criterion = DiceBCELoss()

def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    freeze_backbone_stats=False,
    description=None,
):
    model.train()

    # Frozen parameters alone are not enough:
    # keep pretrained backbone BatchNorm running statistics unchanged.
    if freeze_backbone_stats:
        model.backbone.eval()

    losses = []

    for images, targets, _ in tqdm(
        loader,
        desc=description,
        leave=False,
    ):
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=DEVICE.type,
            enabled=DEVICE.type == "cuda",
        ):
            logits = model(images)
            loss = criterion(
                logits,
                targets,
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        losses.append(
            float(loss.item())
        )

    return float(np.mean(losses))


## Augmented cross-validation run

Separate run directory (`mobilenetv3_unet_cv5_augv2`), same 5-fold split and architecture as above — only the transform pipeline and its loader are specific to this run.

In [ ]:
# AUGV2 CELL 1 — Padding-safe resize + stronger augmentation pipeline
#
# Key change vs the original train_transform: LongestMaxSize + PadIfNeeded
# preserves aspect ratio (pads to square) instead of Resize's hard warp.
# Also adds scale jitter, crop, small rotation/shear, and coarse dropout.

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

PAD_VALUE = 0          # image padding fill (black)
MASK_PAD_VALUE = 0     # mask padding fill (background / not-floor)

train_transform_v2 = A.Compose([
    A.LongestMaxSize(max_size=IMAGE_SIZE),
    A.PadIfNeeded(
        min_height=IMAGE_SIZE,
        min_width=IMAGE_SIZE,
        border_mode=cv2.BORDER_CONSTANT,
        value=PAD_VALUE,
        mask_value=MASK_PAD_VALUE,
    ),
    A.RandomScale(scale_limit=0.15, p=0.5),
    # RandomScale can change output size, so pad back up before cropping.
    A.PadIfNeeded(
        min_height=IMAGE_SIZE,
        min_width=IMAGE_SIZE,
        border_mode=cv2.BORDER_CONSTANT,
        value=PAD_VALUE,
        mask_value=MASK_PAD_VALUE,
    ),
    A.RandomCrop(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Affine(
        rotate=(-8, 8),
        shear=(-5, 5),
        p=0.3,
        mode=cv2.BORDER_CONSTANT,
        cval=PAD_VALUE,
        cval_mask=MASK_PAD_VALUE,
    ),
    A.RandomBrightnessContrast(p=0.35),
    A.HueSaturationValue(p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.1),
    A.CoarseDropout(
        num_holes_range=(1, 4),
        hole_height_range=(0.05, 0.15),
        hole_width_range=(0.05, 0.15),
        fill_value=PAD_VALUE,
        mask_fill_value=None,  # leave mask holes untouched (only occlude the image)
        p=0.15,
    ),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
    ToTensorV2(),
])

# Validation stays deterministic: same padding-safe resize, no random augmentation.
val_transform_v2 = A.Compose([
    A.LongestMaxSize(max_size=IMAGE_SIZE),
    A.PadIfNeeded(
        min_height=IMAGE_SIZE,
        min_width=IMAGE_SIZE,
        border_mode=cv2.BORDER_CONSTANT,
        value=PAD_VALUE,
        mask_value=MASK_PAD_VALUE,
    ),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
    ToTensorV2(),
])

print("train_transform_v2 and val_transform_v2 created.")


In [ ]:
# AUGV2 CELL 2 — Loader function that takes transforms as arguments
# (does not read the global train_transform/val_transform, so it cannot
# collide with the baseline run above).

def make_loaders_v2(manifest, fold, train_tf, val_tf, batch_size=None, num_workers=None):
    batch_size = BATCH_SIZE if batch_size is None else batch_size
    num_workers = NUM_WORKERS if num_workers is None else num_workers

    train_rows = manifest[manifest["fold"] != fold].reset_index(drop=True)
    val_rows = manifest[manifest["fold"] == fold].reset_index(drop=True)

    train_ds = FloorDataset(LOCAL_DATA, train_rows, train_tf)
    val_ds = FloorDataset(LOCAL_DATA, val_rows, val_tf)

    generator = torch.Generator()
    generator.manual_seed(SEED + fold)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=num_workers,
        pin_memory=PIN_MEMORY,
        drop_last=True,
        persistent_workers=num_workers > 0,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        persistent_workers=num_workers > 0,
    )

    return train_loader, val_loader, train_rows, val_rows

print("make_loaders_v2 ready.")


In [ ]:
# Configuration (separate run folder for this experiment)

MOBILE_AUGV2_ARCHITECTURE = "MobileNetV3-UNet-AugV2"
MOBILE_AUGV2_RUN_NAME = "mobilenetv3_unet_cv5_augv2"
MOBILE_AUGV2_RUN_DIR = DRIVE_ROOT / "runs" / MOBILE_AUGV2_RUN_NAME
MOBILE_AUGV2_RUN_DIR.mkdir(parents=True, exist_ok=True)

MOBILE_AUGV2_FOLDS = list(range(N_SPLITS))

# Reuses the shared schedule/LR values defined above.
MOBILE_AUGV2_FROZEN_EPOCHS = FROZEN_EPOCHS
MOBILE_AUGV2_FINETUNE_EPOCHS = FINETUNE_EPOCHS
MOBILE_AUGV2_WARMUP_LR = DECODER_WARMUP_LR
MOBILE_AUGV2_BACKBONE_LR = BACKBONE_FINETUNE_LR
MOBILE_AUGV2_DECODER_LR = DECODER_FINETUNE_LR
MOBILE_AUGV2_WEIGHT_DECAY = WEIGHT_DECAY
MOBILE_AUGV2_PATIENCE = EARLY_STOPPING_PATIENCE

print("Run directory:", MOBILE_AUGV2_RUN_DIR)
print("Folds:", MOBILE_AUGV2_FOLDS)


In [ ]:
# MOBILENET AUGV2 CELL 2 — Fold training with resume support (checkpoint every epoch)

def save_json_v2(data, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with open(destination, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2)


def run_mobile_augv2_fold(fold):
    seed_everything(SEED + fold)

    fold_dir = MOBILE_AUGV2_RUN_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    best_path = fold_dir / "best.pt"
    last_path = fold_dir / "last.pt"
    history_path = fold_dir / "history.csv"
    summary_path = fold_dir / "summary.json"

    if summary_path.exists() and best_path.exists():
        print(f"MobileNet augv2 fold {fold} already completed; skipping.")
        with open(summary_path, "r", encoding="utf-8") as file:
            return json.load(file)

    train_loader, val_loader, train_rows, val_rows = make_loaders_v2(
        manifest, fold, train_transform_v2, val_transform_v2,
    )

    print(f"\nMobileNet augv2 fold {fold}: train={len(train_rows)}, val={len(val_rows)}")

    model = MobileNetV3UNet(pretrained=True).to(DEVICE)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

    history = []
    best_iou = -1.0
    best_epoch = -1
    best_metrics = None
    patience_counter = 0
    start_epoch = 1
    resume_state = None

    if last_path.exists():
        print("Resuming interrupted fold from:", last_path)
        resume_state = torch.load(last_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(resume_state["model"])

        scaler_state = resume_state.get("scaler")
        if scaler_state is not None:
            scaler.load_state_dict(scaler_state)

        start_epoch = int(resume_state["epoch"]) + 1
        best_iou = float(resume_state["best_iou"])
        best_epoch = int(resume_state["best_epoch"])
        best_metrics = resume_state.get("best_metrics")
        patience_counter = int(resume_state.get("patience_counter", 0))
        history = resume_state.get("history", [])
        print("Continuing at global epoch:", start_epoch)

        if resume_state.get("backbone_frozen"):
            model.freeze_backbone()

    # ---------------- Phase 1: frozen pretrained encoder ----------------
    if start_epoch <= MOBILE_AUGV2_FROZEN_EPOCHS:
        model.freeze_backbone()

        optimizer = torch.optim.AdamW(
            list(model.decoder_parameters()),
            lr=MOBILE_AUGV2_WARMUP_LR,
            weight_decay=MOBILE_AUGV2_WEIGHT_DECAY,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(MOBILE_AUGV2_FROZEN_EPOCHS, 1),
        )

        if resume_state is not None and resume_state.get("phase") == "decoder_warmup":
            optimizer.load_state_dict(resume_state["optimizer"])
            scheduler.load_state_dict(resume_state["scheduler"])

        for epoch in range(start_epoch, MOBILE_AUGV2_FROZEN_EPOCHS + 1):
            train_loss = train_one_epoch(
                model, train_loader, optimizer, scaler,
                freeze_backbone_stats=True,
                description=f"Fold {fold} augv2 warm-up {epoch}/{MOBILE_AUGV2_FROZEN_EPOCHS}",
            )
            metrics = evaluate(model, val_loader, criterion)
            scheduler.step()

            row = {"fold": fold, "epoch": epoch, "phase": "decoder_warmup",
                   "train_loss": train_loss, **metrics}
            history.append(row)
            print(row)

            if metrics["iou"] > best_iou:
                best_iou = metrics["iou"]
                best_metrics = metrics
                best_epoch = epoch
                patience_counter = 0
                torch.save(
                    {"model": model.state_dict(), "fold": fold, "epoch": epoch,
                     "phase": "decoder_warmup", "metrics": metrics},
                    best_path,
                )
            else:
                patience_counter += 1

            pd.DataFrame(history).to_csv(history_path, index=False)

            torch.save(
                {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                 "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                 "fold": fold, "epoch": epoch, "phase": "decoder_warmup",
                 "best_iou": best_iou, "best_epoch": best_epoch,
                 "best_metrics": best_metrics, "patience_counter": patience_counter,
                 "history": history, "backbone_frozen": True},
                last_path,
            )

    # ---------------- Phase 2: unfreeze and fine-tune ----------------
    model.unfreeze_backbone()

    if resume_state is None or resume_state.get("phase") == "decoder_warmup":
        patience_counter = 0

    optimizer = torch.optim.AdamW(
        [
            {"params": model.backbone.parameters(), "lr": MOBILE_AUGV2_BACKBONE_LR},
            {"params": list(model.decoder_parameters()), "lr": MOBILE_AUGV2_DECODER_LR},
        ],
        weight_decay=MOBILE_AUGV2_WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(MOBILE_AUGV2_FINETUNE_EPOCHS, 1),
    )

    if resume_state is not None and resume_state.get("phase") == "full_finetune":
        optimizer.load_state_dict(resume_state["optimizer"])
        scheduler.load_state_dict(resume_state["scheduler"])

    first_finetune_epoch = max(start_epoch, MOBILE_AUGV2_FROZEN_EPOCHS + 1)
    final_epoch = MOBILE_AUGV2_FROZEN_EPOCHS + MOBILE_AUGV2_FINETUNE_EPOCHS

    for global_epoch in range(first_finetune_epoch, final_epoch + 1):
        phase_epoch = global_epoch - MOBILE_AUGV2_FROZEN_EPOCHS

        train_loss = train_one_epoch(
            model, train_loader, optimizer, scaler,
            freeze_backbone_stats=False,
            description=f"Fold {fold} augv2 fine-tune {phase_epoch}/{MOBILE_AUGV2_FINETUNE_EPOCHS}",
        )
        metrics = evaluate(model, val_loader, criterion)
        scheduler.step()

        row = {"fold": fold, "epoch": global_epoch, "phase": "full_finetune",
               "train_loss": train_loss, **metrics}
        history.append(row)
        print(row)

        if metrics["iou"] > best_iou:
            best_iou = metrics["iou"]
            best_metrics = metrics
            best_epoch = global_epoch
            patience_counter = 0
            torch.save(
                {"model": model.state_dict(), "fold": fold, "epoch": global_epoch,
                 "phase": "full_finetune", "metrics": metrics},
                best_path,
            )
        else:
            patience_counter += 1

        pd.DataFrame(history).to_csv(history_path, index=False)

        torch.save(
            {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
             "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
             "fold": fold, "epoch": global_epoch, "phase": "full_finetune",
             "best_iou": best_iou, "best_epoch": best_epoch,
             "best_metrics": best_metrics, "patience_counter": patience_counter,
             "history": history, "backbone_frozen": False},
            last_path,
        )

        if patience_counter >= MOBILE_AUGV2_PATIENCE:
            print(f"Early stopping fold {fold} at epoch {global_epoch}.")
            break

    if best_metrics is None:
        raise RuntimeError(f"No valid checkpoint produced for fold {fold}.")

    summary = {
        "architecture": MOBILE_AUGV2_ARCHITECTURE,
        "run_name": MOBILE_AUGV2_RUN_NAME,
        "fold": int(fold),
        "train_samples": int(len(train_rows)),
        "val_samples": int(len(val_rows)),
        "best_epoch": int(best_epoch),
        "best_iou": float(best_metrics["iou"]),
        "best_dice": float(best_metrics["dice"]),
        "best_precision": float(best_metrics["precision"]),
        "best_recall": float(best_metrics["recall"]),
        "frozen_epochs": int(MOBILE_AUGV2_FROZEN_EPOCHS),
        "maximum_finetune_epochs": int(MOBILE_AUGV2_FINETUNE_EPOCHS),
    }

    save_json_v2(summary, summary_path)

    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary

print("run_mobile_augv2_fold ready.")


In [ ]:
# MOBILENET AUGV2 CELL 3 — Run all folds (safe to leave running unattended;
# re-running this cell after a disconnect resumes mid-fold, and skips finished folds)

mobile_augv2_summaries = []

for fold in MOBILE_AUGV2_FOLDS:
    try:
        summary = run_mobile_augv2_fold(fold)
    except Exception as exc:
        print(f"MobileNet augv2 fold {fold} raised an error: {exc}")
        print("Re-run this cell to resume from the last saved checkpoint.")
        raise
    mobile_augv2_summaries.append(summary)

print("All MobileNet augv2 folds complete.")


In [ ]:
# MOBILENET AUGV2 CELL 4 — Aggregate results

mobile_augv2_fold_summaries = []
for fold in MOBILE_AUGV2_FOLDS:
    summary_path = MOBILE_AUGV2_RUN_DIR / f"fold_{fold}" / "summary.json"
    if summary_path.exists():
        with open(summary_path, "r", encoding="utf-8") as file:
            mobile_augv2_fold_summaries.append(json.load(file))

assert mobile_augv2_fold_summaries, "No completed MobileNet augv2 fold summaries found."

mobile_augv2_results = pd.DataFrame(mobile_augv2_fold_summaries).sort_values("fold")
display(mobile_augv2_results)

metric_columns = ["best_iou", "best_dice", "best_precision", "best_recall"]
mobile_augv2_aggregate = {
    "architecture": MOBILE_AUGV2_ARCHITECTURE,
    "run_name": MOBILE_AUGV2_RUN_NAME,
    "n_splits": N_SPLITS,
    "completed_folds": int(len(mobile_augv2_results)),
    "mean": {c: float(mobile_augv2_results[c].mean()) for c in metric_columns},
    "std": {c: float(mobile_augv2_results[c].std(ddof=1)) if len(mobile_augv2_results) > 1 else 0.0
            for c in metric_columns},
}

save_json_v2(mobile_augv2_aggregate, MOBILE_AUGV2_RUN_DIR / "cv_summary.json")

print(json.dumps(mobile_augv2_aggregate["mean"], indent=2))
print(json.dumps(mobile_augv2_aggregate["std"], indent=2))
